# Notebook 03 — Fine-tune viBERT cho phân loại spam/ham

Chạy trên **Google Colab** (Runtime → Change runtime type → GPU; T4 là đủ, GPU mạnh hơn sẽ nhanh hơn).

**Trước khi chạy**
1. Vào trang dataset `p-storm/ptit-facebook-comments-spamham` trên Hugging Face, đăng nhập và **chấp nhận điều kiện truy cập** (dataset ở chế độ gated).
2. Tạo Access Token (quyền *read*) và thêm vào Colab Secrets với tên `HF_TOKEN` (biểu tượng chìa khóa bên trái), bật *Notebook access*. Nếu không có secret, notebook sẽ hiện ô đăng nhập.

**Thiết kế thực nghiệm**
- Chia dữ liệu giống notebook 02: khóa `norm_key(cleaned_text)`, `bucket(key, "test") == 0` làm test (~20%), nhóm bình luận trùng nội dung luôn nằm cùng một phía.
- Validation (~10% của train) cũng chia theo nhóm hash. Epoch, ngưỡng và mô hình tốt nhất **chỉ chọn trên validation**; test chỉ dùng để báo cáo.
- Dữ liệu là `labeled_v7.csv` (38.233 mẫu) — cùng v7 của notebook 02, nên test set trùng (kỳ vọng 7.474 mẫu). Baseline TF-IDF + SVM vẫn được huấn luyện lại trên đúng phần train này để so sánh trực tiếp.

## 1. Cài đặt và cấu hình

Cell cài đặt ghim `transformers<5` vì bản 5 gây lỗi dựng tokenizer. **Nếu Colab yêu cầu restart runtime, hãy restart rồi chạy lại từ đầu.**

In [ ]:
!pip install -U "transformers>=4.44,<5" accelerate huggingface_hub

In [ ]:
import os, re, json, hashlib, unicodedata, random, time, platform
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

OUT = Path("/content/outputs")
OUT.mkdir(parents=True, exist_ok=True)

REPO_ID = "p-storm/ptit-facebook-comments-spamham"
FILENAME = "labeled_v7.csv"   # tên thật trong repo (trang mô tả ghi sai)

# Siêu tham số huấn luyện
SEEDS = [42, 43, 44]       # đặt [42] để chạy nhanh
MAX_LEN = 128              # kiểm tra lại ở mục 4
BATCH = 32
EPOCHS = 4
LR = 2e-5
WARMUP = 0.1
WEIGHT_DECAY = 0.01
PATIENCE = 2               # dừng sớm nếu F1 spam trên validation không tăng

MODELS = {
    "vibert": {"name": "NlpHUST/vibert4news-base-cased", "segment": False},
}
# (mô hình, dùng loss có trọng số lớp?)
RUNS = [("vibert", False)]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Chưa bật GPU: Runtime → Change runtime type → GPU."
print(torch.__version__, torch.cuda.get_device_name(0))

## 2. Nạp dữ liệu từ Hugging Face

In [ ]:
import getpass
from huggingface_hub import login, hf_hub_download

# Có thể dán trực tiếp token vào giữa cặp nháy kép bên dưới nếu không muốn nhập qua hộp thoại VS Code:
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Nhập HF_TOKEN (Hugging Face Read Token) rồi nhấn Enter: ").strip()

assert HF_TOKEN, "Bạn chưa nhập HF_TOKEN!"
login(token=HF_TOKEN)
print("Đăng nhập Hugging Face thành công!")

In [ ]:
from huggingface_hub import list_repo_files
files = list_repo_files(REPO_ID, repo_type="dataset")
print("File trong repo:", files)
if FILENAME not in files:
    csvs = [f for f in files if f.lower().endswith(".csv")]
    assert csvs, "Repo không có file .csv"
    FILENAME = csvs[0]
    print("Không thấy tên mặc định, dùng:", FILENAME)

csv_path = hf_hub_download(REPO_ID, FILENAME, repo_type="dataset")
raw = pd.read_csv(csv_path)
print(raw.shape)
print(raw.columns.tolist())

need = {"id", "label", "cleaned_text", "text"}
assert need <= set(raw.columns), f"Thiếu cột: {need - set(raw.columns)}"

raw["label"] = raw["label"].astype(str).str.strip().str.lower()
assert set(raw["label"].unique()) <= {"spam", "ham"}, raw["label"].unique()
raw = raw.dropna(subset=["cleaned_text"]).copy()
raw["cleaned_text"] = raw["cleaned_text"].astype(str)
raw = raw[raw["cleaned_text"].str.strip() != ""].reset_index(drop=True)
raw["y"] = (raw["label"] == "spam").astype(int)

print("Số mẫu:", len(raw), "| spam:", int(raw.y.sum()), f"({raw.y.mean():.2%})")
print("ID trùng:", int(raw["id"].duplicated().sum()))
print("Kỳ vọng (v7): 38.233 mẫu, 5.326 spam (13,93%), 0 ID trùng.")
raw.sample(5, random_state=0)[["label", "cleaned_text"]]

## 3. Chia train / validation / test

Dùng đúng `norm_key` và `bucket` của `pipeline/modeling/training_utils.py` để split trùng logic với notebook 02.

In [ ]:
def norm_key(text):
    normalized = unicodedata.normalize("NFC", str(text)).lower()
    normalized = re.sub(r"\s+", " ", normalized).strip()
    key = re.sub(r"\d+", "0", normalized)
    key = re.sub(r"[^\w\s]", " ", key)
    key = re.sub(r"\s+", " ", key).strip()
    return key or "__symbols__|" + normalized

def bucket(key, salt, n=5):
    return int(hashlib.md5(f"{salt}|{key}".encode()).hexdigest(), 16) % n

keys = raw["cleaned_text"].map(norm_key).to_numpy()
is_test = np.array([bucket(k, "test") == 0 for k in keys])
is_val = np.array([(not t) and bucket(k, "val", 10) == 0 for k, t in zip(keys, is_test)])
is_tr = ~is_test & ~is_val

idx_tr, idx_va, idx_te = np.where(is_tr)[0], np.where(is_val)[0], np.where(is_test)[0]
assert not set(keys[idx_tr]) & set(keys[idx_va])
assert not set(keys[idx_tr]) & set(keys[idx_te])
assert not set(keys[idx_va]) & set(keys[idx_te])

y_all = raw["y"].to_numpy()
y_tr, y_va, y_te = y_all[idx_tr], y_all[idx_va], y_all[idx_te]
for name, y in [("train", y_tr), ("val", y_va), ("test", y_te)]:
    print(f"{name:5s}: {len(y):6d} mẫu | spam {y.mean():.2%}")
print("train+val =", len(idx_tr) + len(idx_va), "(kỳ vọng 30759) | test =", len(idx_te), "(kỳ vọng 7474)")
if (len(idx_tr) + len(idx_va), len(idx_te)) != (30759, 7474):
    print("CẢNH BÁO: số mẫu khác notebook 02, test set có thể không trùng. Hãy kiểm tra lại dữ liệu.")

## 4. Chuẩn bị văn bản

- **viBERT**: dùng thẳng `cleaned_text` (đã có `urltoken`/`phonetoken`), vì WordPiece tự xử lý.
- **Baseline** TF-IDF + SVM dùng `cleaned_text` chữ thường.

In [ ]:
raw["seg"] = raw["cleaned_text"]
raw[["cleaned_text"]].head(3)

In [ ]:
from transformers import AutoTokenizer, BertTokenizerFast

def load_tok(name):
    errors = []
    local = Path(name).exists()
    if "vibert" in name.lower() and not local:
        # viBERT chỉ có vocab.txt (WordPiece): dựng trực tiếp, không cần sentencepiece
        try:
            vocab = hf_hub_download(name, "vocab.txt")
            return BertTokenizerFast(vocab_file=vocab, do_lower_case=False)
        except Exception as e:
            errors.append(("BertTokenizerFast", repr(e)[:300]))
    for kw in ({}, {"use_fast": False}):
        try:
            return AutoTokenizer.from_pretrained(name, **kw)
        except Exception as e:
            errors.append((str(kw), repr(e)[:300]))
    for item in errors:
        print(item)
    raise RuntimeError(f"Không nạp được tokenizer cho {name}")

# Kiểm tra sớm: mọi mô hình trong RUNS phải nạp được tokenizer trước khi huấn luyện
for key in sorted({m for m, _ in RUNS}):
    t = load_tok(MODELS[key]["name"])
    sample = raw["seg" if MODELS[key]["segment"] else "cleaned_text"].iloc[0]
    print(f"{key:8s} -> {type(t).__name__:22s} {t(sample)['input_ids'][:12]}")

tok_probe = load_tok(MODELS["vibert"]["name"])
lens = [len(x) for x in tok_probe(raw["cleaned_text"].iloc[idx_tr].tolist(), add_special_tokens=True)["input_ids"]]
print("Độ dài token (viBERT) — p50/p90/p95/p99/max:", np.percentile(lens, [50, 90, 95, 99]).astype(int).tolist(), max(lens))
print(f"Tỷ lệ bị cắt ở MAX_LEN={MAX_LEN}: {np.mean(np.array(lens) > MAX_LEN):.2%}")

## 5. Baseline TF-IDF + SVM trên cùng split

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, confusion_matrix)

def metrics(y, pred, score):
    return {"Accuracy": accuracy_score(y, pred),
            "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred, zero_division=0),
            "F1": f1_score(y, pred, zero_division=0),
            "AP": average_precision_score(y, score)}

seg_lower = raw["seg"].str.lower().to_numpy()
base = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), LinearSVC())
base.fit(seg_lower[idx_tr], y_tr)
base_score = base.decision_function(seg_lower[idx_te])
base_pred = base.predict(seg_lower[idx_te])
base_metrics = metrics(y_te, base_pred, base_score)
print("Baseline TF-IDF + SVM (test):", {k: round(v, 4) for k, v in base_metrics.items()})
print("Tham chiếu notebook 02 (cùng test set, huấn luyện trên 30.759 mẫu và tinh chỉnh tham số): F1 0,7334 | AP 0,8117 | Recall 62,78%.")

## 6. Hàm huấn luyện

- Loss: cross-entropy.
- Mỗi epoch: tính F1 spam **tốt nhất theo ngưỡng** trên validation; giữ epoch tốt nhất và dừng sớm.
- Ngưỡng cuối cùng chọn trên validation, áp lên test đúng một lần.

In [ ]:
from transformers import AutoModelForSequenceClassification, BertForSequenceClassification, get_linear_schedule_with_warmup

use_amp = device.type == "cuda"
# QUAN TRỌNG: Tesla T4 (compute capability 7.5) KHÔNG có phần cứng bfloat16 (torch.cuda.is_bf16_supported() báo True do giả lập phần mềm làm chậm gấp 18 lần!).
# Chỉ dùng bfloat16 trên GPU Ampere/Ada trở lên (compute capability >= 8.0 như A100/L4), còn T4 bắt buộc dùng float16 Tensor Cores:
cc_major = torch.cuda.get_device_capability(0)[0] if use_amp else 0
amp_dtype = torch.bfloat16 if (use_amp and cc_major >= 8) else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(use_amp and amp_dtype == torch.float16))
TRAIN_BS = BATCH
EVAL_BS = 128
print(f"GPU: {torch.cuda.get_device_name(0)} (cc {cc_major}.x) | dtype: {amp_dtype} (Hardware Tensor Cores) | TRAIN_BS={TRAIN_BS} | EVAL_BS={EVAL_BS}")

def load_cls_model(name, num_labels=2):
    # NlpHUST/vibert4news-base-cased ghi nhầm model_type='ibert' trong config.json nhưng trọng số là bert.*
    if "vibert" in name.lower():
        return BertForSequenceClassification.from_pretrained(name, num_labels=num_labels)
    return AutoModelForSequenceClassification.from_pretrained(name, num_labels=num_labels)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

class DS(torch.utils.data.Dataset):
    def __init__(self, enc, y):
        self.enc, self.y, self.keys = enc, y, list(enc.keys())
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        item = {k: self.enc[k][i] for k in self.keys}
        item["labels"] = int(self.y[i])
        return item

_cache = {}
def get_data(model_key):
    if model_key in _cache:
        return _cache[model_key]
    cfg = MODELS[model_key]
    tok = load_tok(cfg["name"])
    col = "seg" if cfg["segment"] else "cleaned_text"
    texts = raw[col].to_numpy()
    def enc(idx):
        return tok(texts[idx].tolist(), truncation=True, max_length=MAX_LEN)
    data = {"tok": tok,
            "train": DS(enc(idx_tr), y_tr), "val": DS(enc(idx_va), y_va), "test": DS(enc(idx_te), y_te)}
    _cache[model_key] = data
    return data

def best_threshold(y, p):
    grid = np.linspace(0.05, 0.95, 181)
    scores = [f1_score(y, p >= t, zero_division=0) for t in grid]
    i = int(np.argmax(scores))
    return float(grid[i]), float(scores[i])

@torch.no_grad()
def predict_proba(model, ds, tok, bs=None):
    if bs is None:
        bs = EVAL_BS
    model.eval()
    out = []
    dl = DataLoader(ds, batch_size=bs, pin_memory=use_amp, collate_fn=lambda f: tok.pad(f, return_tensors="pt"))
    for b in dl:
        b = {k: v.to(device, non_blocking=True) for k, v in b.items() if k != "labels"}
        with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
            logits = model(**b).logits
        out.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu())
    return torch.cat(out).numpy()

ALL = []                      # kết quả mọi lần chạy
BEST = {"val_f1": -1.0}       # lần chạy tốt nhất theo validation

def run_one(model_key, weighted, seed):
    cfg, data = MODELS[model_key], get_data(model_key)
    tok = data["tok"]
    run_name = f"{model_key}{'+weight' if weighted else ''}"
    print(f"\n=== {run_name} | seed {seed} ===")
    set_seed(seed)
    model = load_cls_model(cfg["name"], num_labels=2).to(device)

    g = torch.Generator(); g.manual_seed(seed)
    dl = DataLoader(data["train"], batch_size=TRAIN_BS, shuffle=True, generator=g, pin_memory=use_amp,
                    collate_fn=lambda f: tok.pad(f, return_tensors="pt"))
    steps = EPOCHS * len(dl)
    no_decay = ("bias", "LayerNorm.weight")
    groups = [
        {"params": [p for n, p in model.named_parameters() if not any(x in n for x in no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if any(x in n for x in no_decay)], "weight_decay": 0.0},
    ]
    opt = torch.optim.AdamW(groups, lr=LR, fused=use_amp)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    w = torch.tensor([1.0, float(np.sqrt((y_tr == 0).sum() / (y_tr == 1).sum()))], device=device) if weighted else None
    loss_fn = torch.nn.CrossEntropyLoss(weight=w)

    best, bad = {"f1": -1.0}, 0
    for epoch in range(1, EPOCHS + 1):
        model.train(); t0 = time.time(); total = 0.0
        for batch in dl:
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
                logits = model(**batch).logits
            loss = loss_fn(logits.float(), labels)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            total += loss.item()
        pv = predict_proba(model, data["val"], tok)
        thr, f1v = best_threshold(y_va, pv)
        apv = average_precision_score(y_va, pv)
        print(f"epoch {epoch}: loss {total/len(dl):.4f} | val F1* {f1v:.4f} (thr {thr:.2f}) | val AP {apv:.4f} | {time.time()-t0:.0f}s")
        if f1v > best["f1"]:
            best = {"f1": f1v, "thr": thr, "epoch": epoch, "pv": pv, "ap": apv,
                    "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print("Dừng sớm."); break

    model.load_state_dict(best["state"])
    pt = predict_proba(model, data["test"], tok)
    pred = (pt >= best["thr"]).astype(int)
    res = {"run": run_name, "seed": seed, "epoch": best["epoch"], "thr": best["thr"],
           "val_F1": best["f1"], "val_AP": best["ap"], **metrics(y_te, pred, pt),
           "_pt": pt, "_pred": pred}
    print("TEST:", {k: round(v, 4) for k, v in res.items() if k in ("Accuracy", "Precision", "Recall", "F1", "AP")})
    ALL.append(res)

    if best["f1"] > BEST["val_f1"]:                      # chọn bản lưu theo validation
        BEST.update({"val_f1": best["f1"], "run": run_name, "seed": seed, "thr": best["thr"]})
        d = OUT / "best_model"
        model.save_pretrained(d); tok.save_pretrained(d)
        (d / "threshold.json").write_text(json.dumps(
            {"threshold": best["thr"], "model": cfg["name"], "segment": cfg["segment"],
             "max_len": MAX_LEN, "run": run_name, "seed": seed}, indent=2))
    del model, opt, best
    torch.cuda.empty_cache()

## 7. Chạy thực nghiệm

Mỗi lần chạy khoảng vài phút trên GPU T4; tổng thời gian phụ thuộc số seed và số cấu hình trong `RUNS`.

In [ ]:
for key in sorted({m for m, _ in RUNS}):
    get_data(key)            # mã hóa trước; lỗi tokenizer sẽ hiện ngay, trước khi tốn thời gian GPU
for model_key, weighted in RUNS:
    for seed in SEEDS:
        run_one(model_key, weighted, seed)

## 8. Kết quả

In [ ]:
res_df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in ALL])
cols = ["Accuracy", "Precision", "Recall", "F1", "AP"]
print("Chi tiết từng lần chạy:")
display(res_df.round(4))

summary = res_df.groupby("run")[cols + ["val_F1"]].agg(["mean", "std"]).round(4)
print("\nTrung bình ± độ lệch chuẩn theo seed:")
display(summary)

base_row = pd.DataFrame([base_metrics], index=["TF-IDF + SVM (baseline, chạy lại)"]).round(4)
print("\nBaseline:")
display(base_row)

In [ ]:
# Chọn mô hình theo VALIDATION, không theo test
best_res = max(ALL, key=lambda r: r["val_F1"])
print(f"Mô hình tốt nhất theo validation: {best_res['run']} seed {best_res['seed']} "
      f"(val F1* {best_res['val_F1']:.4f}, ngưỡng {best_res['thr']:.2f})")

def boot_ci(y, pred, n=500, seed=0):
    rng = np.random.default_rng(seed)
    v = [f1_score(y[i], pred[i], zero_division=0) for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(v, [2.5, 97.5])

def paired_bootstrap(y, a, b, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        d.append(f1_score(y[i], a[i], zero_division=0) - f1_score(y[i], b[i], zero_division=0))
    d = np.array(d)
    return float(d.mean()), np.percentile(d, [2.5, 97.5]), float((d <= 0).mean())

ci_b = boot_ci(y_te, base_pred); ci_m = boot_ci(y_te, best_res["_pred"])
print(f"F1 spam baseline : {base_metrics['F1']:.4f}  CI95% [{ci_b[0]:.4f}, {ci_b[1]:.4f}]")
print(f"F1 spam {best_res['run']:<9s}: {best_res['F1']:.4f}  CI95% [{ci_m[0]:.4f}, {ci_m[1]:.4f}]")
mean_d, ci_d, p_le0 = paired_bootstrap(y_te, best_res["_pred"], base_pred)
print(f"Chênh lệch F1 (mô hình − baseline): {mean_d:+.4f}, CI95% [{ci_d[0]:+.4f}, {ci_d[1]:+.4f}], P(chênh lệch ≤ 0) ≈ {p_le0:.3f}")

for name, pred in [("Baseline", base_pred), (best_res["run"], best_res["_pred"])]:
    cm = confusion_matrix(y_te, pred, labels=[0, 1])
    print(f"\nConfusion matrix — {name}")
    display(pd.DataFrame(cm, index=["thật: ham", "thật: spam"], columns=["đoán: ham", "đoán: spam"]))

## 9. Phân tích lỗi

Xem các mẫu mô hình tốt nhất đoán sai, và các mẫu hai mô hình bất đồng. Cột hiển thị là văn bản gốc `text`.

In [ ]:
te = raw.iloc[idx_te].reset_index(drop=True).copy()
te["base_pred"], te["model_pred"], te["model_prob"] = base_pred, best_res["_pred"], best_res["_pt"]
pd.set_option("display.max_colwidth", 160)

fn = te[(te.y == 1) & (te.model_pred == 0)]
fp = te[(te.y == 0) & (te.model_pred == 1)]
print(f"FN (bỏ sót spam): {len(fn)} | FP (ham bị báo spam): {len(fp)}")
print("\n--- 10 FN có điểm spam cao nhất ---")
display(fn.sort_values("model_prob", ascending=False)[["id", "text", "model_prob"]].head(10))
print("\n--- 10 FP có điểm spam cao nhất ---")
display(fp.sort_values("model_prob", ascending=False)[["id", "text", "model_prob"]].head(10))

only_model = te[(te.model_pred == te.y) & (te.base_pred != te.y)]
only_base = te[(te.base_pred == te.y) & (te.model_pred != te.y)]
print(f"\nMô hình đúng, baseline sai: {len(only_model)} | Baseline đúng, mô hình sai: {len(only_base)}")
print("\n--- Spam mà mô hình bắt được nhưng baseline bỏ sót ---")
display(only_model[only_model.y == 1][["id", "text"]].head(10))

te["len_words"] = te["cleaned_text"].str.split().str.len()
te["len_bin"] = pd.cut(te["len_words"], [0, 3, 6, 12, 10_000], labels=["1-3", "4-6", "7-12", ">12"])
by_len = te.groupby("len_bin", observed=True).apply(lambda d: pd.Series({
    "n": len(d), "spam": int(d.y.sum()),
    "F1_model": f1_score(d.y, d.model_pred, zero_division=0),
    "F1_baseline": f1_score(d.y, d.base_pred, zero_division=0)}))
print("\nF1 spam theo độ dài bình luận (số từ):")
display(by_len.round(3))

## 10. Lưu kết quả

In [ ]:
import zipfile, importlib.metadata as md_
res_df.to_csv(OUT / "metrics_per_run.csv", index=False)
summary.to_csv(OUT / "metrics_summary.csv")
pd.DataFrame([base_metrics]).to_csv(OUT / "metrics_baseline.csv", index=False)
te[["id", "y", "base_pred", "model_pred", "model_prob"]].to_csv(OUT / "test_predictions.csv", index=False)

manifest = {
    "dataset": REPO_ID, "file": FILENAME,
    "dataset_sha256": hashlib.sha256(Path(csv_path).read_bytes()).hexdigest(),
    "n_total": int(len(raw)), "n_train": int(len(idx_tr)), "n_val": int(len(idx_va)), "n_test": int(len(idx_te)),
    "config": {"SEEDS": SEEDS, "MAX_LEN": MAX_LEN, "BATCH": BATCH, "EPOCHS": EPOCHS, "LR": LR,
               "WARMUP": WARMUP, "WEIGHT_DECAY": WEIGHT_DECAY, "PATIENCE": PATIENCE, "RUNS": RUNS,
               "MODELS": MODELS},
    "selected_by_validation": {k: BEST[k] for k in ("run", "seed", "thr", "val_f1")},
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": md_.version("transformers"), "gpu": torch.cuda.get_device_name(0)},
}
(OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

zip_path = Path("/content/vibert_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(OUT))
print("Đã nén đầy đủ (gồm cả trọng số best_model):", zip_path, f"({zip_path.stat().st_size/1e6:.1f} MB)")

# Tạo thêm gói gọn nhẹ (chỉ gồm CSV, JSON, threshold.json - ~200 KB) để tải trực tiếp ngay trong VS Code bằng 1 click:
import base64
from IPython.display import HTML, display
light_zip = Path("/content/vibert_report_files.zip")
with zipfile.ZipFile(light_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and p.suffix.lower() in (".csv", ".json", ".txt"):
            z.write(p, p.relative_to(OUT))
b64 = base64.b64encode(light_zip.read_bytes()).decode()
display(HTML(f'<a download="vibert_report_files.zip" href="data:application/zip;base64,{b64}" '
             f'style="display:inline-block;padding:10px 18px;background:#2563eb;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">'
             f'📥 BẤM VÀO ĐÂY ĐỂ TẢI KẾT QUẢ VỀ MÁY (vibert_report_files.zip — {light_zip.stat().st_size/1e3:.1f} KB)</a>'))

# Nếu chạy trên trình duyệt Colab thì tự tải thêm bản full:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

## 11. Demo dự đoán bình luận mới

Dùng mô hình đã chọn theo validation và ngưỡng đã lưu. Văn bản mới phải qua cùng bước làm sạch (URL/SĐT → token) như dữ liệu huấn luyện.

In [ ]:
import html
URL = re.compile(r"https?://\S+|www\.\S+", re.I)
PHONE = re.compile(r"(?<!\d)(?:\+84|0084|84|0)(?:[\s.-]?[3-9](?:[\s.-]?\d){8})(?!\d)")

def light_clean(text):
    text = unicodedata.normalize("NFC", html.unescape(str(text)))
    text = re.sub(r"<[^>]+>", " ", text)
    text = URL.sub(" urltoken ", text)
    text = PHONE.sub(" phonetoken ", text)
    return re.sub(r"\s+", " ", text).strip()

d = OUT / "best_model"
meta = json.loads((d / "threshold.json").read_text())
tok_best = load_tok(str(d))
model_best = (BertForSequenceClassification if "vibert" in meta["model"].lower() else AutoModelForSequenceClassification).from_pretrained(d).to(device).eval()

@torch.no_grad()
def predict_texts(texts):
    cleaned = [light_clean(t) for t in texts]
    b = tok_best(cleaned, truncation=True, max_length=meta["max_len"], padding=True, return_tensors="pt").to(device)
    p = torch.softmax(model_best(**b).logits.float(), dim=-1)[:, 1].cpu().numpy()
    return [("spam" if x >= meta["threshold"] else "ham", round(float(x), 3)) for x in p]

samples = [
    "Mọi người ơi cho mình hỏi lịch thi môn giải tích khi nào vậy ạ",
    "Inbox mình nhận làm bài tập thuê, giá rẻ uy tín 0912345678",
    "Bán áo khoác giá sinh viên, ib zalo nhé https://shopee.vn/abc",
    "Cảm ơn bạn nhiều nha",
]
for s, r in zip(samples, predict_texts(samples)):
    print(r, "|", s)

print("\n✅ Đã lưu gói output vào trong notebook (nút tải ở Mục 10). Đang tự động ngắt kết nối GPU Colab...")
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print("Không thể tự ngắt runtime:", e)